In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import json
import shutil
import hashlib
import importlib.util

import numpy as np
import pandas as pd
import torch

from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel

thu_muc_du_lieu = Path("/content/drive/MyDrive/CMU-MOSI")

tep_manifest = (
    thu_muc_du_lieu / "preprocessing/shared/manifest.csv"
)

tep_module = (
    thu_muc_du_lieu / "src/preprocessing/text_processor.py"
)

assert tep_manifest.is_file(), f"Không tìm thấy: {tep_manifest}"
assert tep_module.is_file(), f"Không tìm thấy: {tep_module}"

print("Đã tìm thấy manifest và module Text.")

Đã tìm thấy manifest và module Text.


In [ ]:
manifest = pd.read_csv(
    tep_manifest,
    dtype={
        "id": "string",
        "video_id": "string",
        "clip_id": "string",
    },
    keep_default_na=False,
)

assert len(manifest) == 2199, "Manifest không đủ 2.199 mẫu"
assert manifest["id"].is_unique, "Có ID trùng"

id_tao_lai = (
    manifest["video_id"] + "$_$" + manifest["clip_id"]
)

assert manifest["id"].eq(id_tao_lai).all(), "Có ID sai"
assert manifest["raw_text"].str.strip().ne("").all(), (
    "Có transcript trống"
)

so_mau_split = manifest["split"].value_counts().to_dict()

assert so_mau_split == {
    "train": 1284,
    "valid": 229,
    "test": 686,
}, "Phân bố split chưa đúng"

nhan = pd.to_numeric(manifest["label"], errors="raise")

assert np.isfinite(nhan.to_numpy()).all(), "Có nhãn không hợp lệ"
assert nhan.between(-3, 3).all(), "Có nhãn ngoài [-3, 3]"

print("Tổng số mẫu:", len(manifest))
print("Phân bố split:", so_mau_split)
print("Số mẫu nhãn 0:", int(nhan.eq(0).sum()))

display(manifest.head())

Tổng số mẫu: 2199
Phân bố split: {'train': 1284, 'test': 686, 'valid': 229}
Số mẫu nhãn 0: 96


,id,video_id,clip_id,split,raw_text,label,source_archive,video_relpath,video_size_bytes
0,03bSnISJMiM$_$11,03bSnISJMiM,11,train,A LOT OF SAD PARTS,-0.5,Raw.zip,Raw/03bSnISJMiM/11.mp4,137288
1,03bSnISJMiM$_$10,03bSnISJMiM,10,train,THERE IS SAD PART,-1.2,Raw.zip,Raw/03bSnISJMiM/10.mp4,101514
2,03bSnISJMiM$_$13,03bSnISJMiM,13,train,AND ITS A REALLY FUNNY,1.8,Raw.zip,Raw/03bSnISJMiM/13.mp4,132253
3,03bSnISJMiM$_$12,03bSnISJMiM,12,train,BUT IT WAS REALLY REALLY AWESOME,2.2,Raw.zip,Raw/03bSnISJMiM/12.mp4,173114
4,03bSnISJMiM$_$1,03bSnISJMiM,1,train,ANYHOW IT WAS REALLY GOOD,2.4,Raw.zip,Raw/03bSnISJMiM/1.mp4,193330


In [ ]:
dac_ta = importlib.util.spec_from_file_location(
    "mosi_text_processor",
    tep_module,
)

text_module = importlib.util.module_from_spec(dac_ta)
dac_ta.loader.exec_module(text_module)

checkpoint = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(
    checkpoint,
    use_fast=True,
)

assert tokenizer.is_fast, "Cần fast tokenizer"

config_text = {
    "checkpoint": checkpoint,
    "tokenizer": tokenizer,
    "extract_embeddings": False,
}

print("Module:", tep_module)
print("Tokenizer:", checkpoint)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Module: /content/drive/MyDrive/CMU-MOSI/src/preprocessing/text_processor.py
Tokenizer: bert-base-uncased


In [ ]:
ma_lan_chay = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%S_%fZ"
)

thu_muc_full = (
    thu_muc_du_lieu
    / "preprocessing"
    / "text"
    / "full"
    / ma_lan_chay
)

thu_muc_embedding = thu_muc_full / "embeddings"

thu_muc_full.mkdir(parents=True, exist_ok=False)
thu_muc_embedding.mkdir()

tep_text_full = thu_muc_full / "text_full_results.json"

manifest.to_csv(
    thu_muc_full / "manifest.csv",
    index=False,
    encoding="utf-8-sig",
)

shutil.copy2(
    tep_module,
    thu_muc_full / "text_processor.py",
)

print("Thư mục kết quả:", thu_muc_full)

Thư mục kết quả: /content/drive/MyDrive/CMU-MOSI/preprocessing/text/full/20261102T152904_636429Z


In [ ]:
ket_qua_full = {}
nhat_ky_text = []

cac_truong_mang = [
    "input_ids",
    "attention_mask",
    "token_type_ids",
    "word_ids",
]

for _, mau in tqdm(
    manifest.iterrows(),
    total=len(manifest),
    desc="Xử lý Text",
):
    id_mau = str(mau["id"])

    try:
        ket_qua = text_module.process_text(
            mau["raw_text"],
            config_text,
        )

        assert ket_qua["raw_text"] == mau["raw_text"]
        assert ket_qua["clean_text"] == " ".join(
            mau["raw_text"].split()
        )
        assert ket_qua["words"] == ket_qua["clean_text"].split()
        assert ket_qua["embeddings"] is None

        so_token = len(ket_qua["input_ids"])

        for truong in cac_truong_mang:
            mang = ket_qua[truong]

            assert mang.ndim == 1, f"{truong} không phải 1 chiều"
            assert mang.dtype == np.int64, f"{truong} sai dtype"
            assert len(mang) == so_token, f"{truong} sai độ dài"

        assert np.all(ket_qua["attention_mask"] == 1)
        assert np.all(ket_qua["token_type_ids"] == 0)

        word_ids = ket_qua["word_ids"]

        assert np.all(
            (word_ids == -1)
            | (
                (word_ids >= 0)
                & (word_ids < len(ket_qua["words"]))
            )
        )

        assert set(word_ids[word_ids >= 0].tolist()) == set(
            range(len(ket_qua["words"]))
        )

        ket_qua_full[id_mau] = {
            truong: (
                gia_tri.tolist()
                if isinstance(gia_tri, np.ndarray)
                else gia_tri
            )
            for truong, gia_tri in ket_qua.items()
        }

        nhat_ky_text.append({
            "id": id_mau,
            "split": mau["split"],
            "status": "success",
            "word_count": len(ket_qua["words"]),
            "token_count": so_token,
            "warning_count": len(ket_qua["warnings"]),
            "error": "",
        })

    except Exception as loi:
        nhat_ky_text.append({
            "id": id_mau,
            "split": mau["split"],
            "status": "failed",
            "error": str(loi),
        })

tep_text_full.write_text(
    json.dumps(
        ket_qua_full,
        ensure_ascii=False,
        indent=2,
        allow_nan=False,
    ),
    encoding="utf-8",
)

bang_text = pd.DataFrame(nhat_ky_text)

bang_text.to_csv(
    thu_muc_full / "text_full_log.csv",
    index=False,
    encoding="utf-8-sig",
)

# Đọc lại để kiểm tra dữ liệu đã lưu
ket_qua_doc_lai = json.loads(
    tep_text_full.read_text(encoding="utf-8")
)

assert ket_qua_doc_lai == ket_qua_full

display(pd.crosstab(bang_text["split"], bang_text["status"]))

mau_loi = bang_text[bang_text["status"] == "failed"]

print("Text thành công:", len(ket_qua_full))
print("Text thất bại:", len(mau_loi))
print("Đã lưu:", tep_text_full)

if not mau_loi.empty:
    display(mau_loi[["id", "error"]])

assert len(ket_qua_full) == 2199, (
    "Có mẫu Text lỗi. Kiểm tra nhật ký trước khi chạy embedding."
)

Xử lý Text:   0%|          | 0/2199 [00:00<?, ?it/s]

status,success
split,
test,686
train,1284
valid,229


Text thành công: 2199
Text thất bại: 0
Đã lưu: /content/drive/MyDrive/CMU-MOSI/preprocessing/text/full/20261102T152904_636429Z/text_full_results.json


In [ ]:
thiet_bi = "cuda" if torch.cuda.is_available() else "cpu"

bert_co_dinh = AutoModel.from_pretrained(checkpoint)
bert_co_dinh.to(thiet_bi)
bert_co_dinh.eval()
bert_co_dinh.requires_grad_(False)

config_embedding = {
    "tokenizer": tokenizer,
    "bert_model": bert_co_dinh,
    "extract_embeddings": True,
}

print("Thiết bị:", thiet_bi)
print("Giới hạn token:", bert_co_dinh.config.max_position_embeddings)
print("Kích thước embedding:", bert_co_dinh.config.hidden_size)

# Kiểm tra trước toàn bộ độ dài, không tự cắt transcript
mau_qua_dai = [
    {
        "id": id_mau,
        "token_count": len(ket_qua["input_ids"]),
    }
    for id_mau, ket_qua in ket_qua_full.items()
    if len(ket_qua["input_ids"])
    > bert_co_dinh.config.max_position_embeddings
]

if mau_qua_dai:
    display(pd.DataFrame(mau_qua_dai))

assert not mau_qua_dai, (
    "Có mẫu vượt giới hạn BERT; cần xử lý riêng, chưa tự cắt."
)

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Thiết bị: cpu
Giới hạn token: 512
Kích thước embedding: 768


In [ ]:
nhat_ky_embedding = []

for _, mau in tqdm(
    manifest.iterrows(),
    total=len(manifest),
    desc="Trích xuất BERT",
):
    id_mau = str(mau["id"])
    ban_text = ket_qua_full[id_mau]
    tep_npz = thu_muc_embedding / f"{id_mau}.npz"

    try:
        ket_qua = text_module.process_text(
            ban_text["raw_text"],
            config_embedding,
        )

        # Bổ sung embedding không được thay đổi Text hoặc token
        for truong in ["raw_text", "clean_text", "words", "warnings"]:
            assert ket_qua[truong] == ban_text[truong], truong

        for truong in cac_truong_mang:
            np.testing.assert_array_equal(
                ket_qua[truong],
                np.asarray(ban_text[truong], dtype=np.int64),
            )

        embedding = ket_qua["embeddings"]

        assert embedding.shape == (
            len(ket_qua["input_ids"]),
            bert_co_dinh.config.hidden_size,
        )
        assert embedding.dtype == np.float32
        assert np.isfinite(embedding).all()

        np.savez_compressed(
            tep_npz,
            embeddings=embedding,
            input_ids=ket_qua["input_ids"],
            attention_mask=ket_qua["attention_mask"],
            token_type_ids=ket_qua["token_type_ids"],
            word_ids=ket_qua["word_ids"],
        )

        # Kiểm tra file vừa lưu
        with np.load(tep_npz, allow_pickle=False) as ban_luu:
            np.testing.assert_array_equal(
                ban_luu["embeddings"],
                embedding,
            )

            for truong in cac_truong_mang:
                np.testing.assert_array_equal(
                    ban_luu[truong],
                    ket_qua[truong],
                )

        nhat_ky_embedding.append({
            "id": id_mau,
            "split": mau["split"],
            "status": "success",
            "token_count": len(ket_qua["input_ids"]),
            "embedding_shape": str(embedding.shape),
            "file": tep_npz.name,
            "error": "",
        })

        del ket_qua, embedding

    except Exception as loi:
        nhat_ky_embedding.append({
            "id": id_mau,
            "split": mau["split"],
            "status": "failed",
            "file": tep_npz.name,
            "error": str(loi),
        })

    # Cập nhật nhật ký sau mỗi mẫu
    pd.DataFrame(nhat_ky_embedding).to_csv(
        thu_muc_full / "embedding_full_log.csv",
        index=False,
        encoding="utf-8-sig",
    )

bang_embedding = pd.DataFrame(nhat_ky_embedding)

display(
    pd.crosstab(
        bang_embedding["split"],
        bang_embedding["status"],
    )
)

mau_loi = bang_embedding[
    bang_embedding["status"] == "failed"
]

if not mau_loi.empty:
    display(mau_loi[["id", "error"]])

print(
    "Embedding thành công:",
    int(bang_embedding["status"].eq("success").sum()),
)
print("Embedding thất bại:", len(mau_loi))

Trích xuất BERT:   0%|          | 0/2199 [00:00<?, ?it/s]

status,success
split,
test,686
train,1284
valid,229


Embedding thành công: 2199
Embedding thất bại: 0


In [ ]:
assert len(bang_embedding) == 2199
assert bang_embedding["id"].is_unique
assert set(bang_embedding["id"]) == set(manifest["id"])
assert bang_embedding["status"].eq("success").all(), (
    "Chưa đủ 2.199 embedding thành công"
)

cac_file_npz = list(thu_muc_embedding.glob("*.npz"))

assert {tep.stem for tep in cac_file_npz} == set(manifest["id"])

cau_hinh = {
    "sample_count": len(manifest),
    "split_counts": so_mau_split,
    "checkpoint": checkpoint,
    "model_revision": getattr(
        bert_co_dinh.config, "_commit_hash", None
    ),
    "cleaning": "normalize_whitespace_only",
    "word_split": "whitespace",
    "padding": False,
    "truncation": False,
    "special_token_word_id": -1,
    "embedding_layer": "last_hidden_state",
    "embedding_level": "token",
    "includes_special_tokens": True,
    "subword_pooling": None,
    "embedding_dtype": "float32",
    "hidden_size": bert_co_dinh.config.hidden_size,
    "device": thiet_bi,
    "deepcu_input_config_confirmed": False,
    "text_results": tep_text_full.name,
    "module_sha256": hashlib.sha256(
        (thu_muc_full / "text_processor.py").read_bytes()
    ).hexdigest(),
    "versions": {
        ten: version(ten)
        for ten in [
            "numpy", "pandas", "torch",
            "transformers", "tokenizers",
        ]
    },
}

(thu_muc_full / "text_full_config.json").write_text(
    json.dumps(cau_hinh, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("HOÀN TẤT PHẦN TEXT")
print("Text:", len(ket_qua_full), "/ 2199")
print("Embedding:", len(cac_file_npz), "/ 2199")
print("Thư mục kết quả:", thu_muc_full)
print("File Text:", tep_text_full)

HOÀN TẤT PHẦN TEXT
Text: 2199 / 2199
Embedding: 2199 / 2199
Thư mục kết quả: /content/drive/MyDrive/CMU-MOSI/preprocessing/text/full/20261102T152904_636429Z
File Text: /content/drive/MyDrive/CMU-MOSI/preprocessing/text/full/20261102T152904_636429Z/text_full_results.json
